# NER Data Augmentation - Baseline Experiment
## Full vs. 10% Subsampled BERT Training on CoNLL-2003

This notebook runs a baseline comparison experiment:
- **Model 1**: BERT fine-tuned on the full CoNLL-2003 training dataset (~14k samples)
- **Model 2**: BERT fine-tuned on 10% subsampled training dataset (~1.4k samples)

Both models are evaluated on the same test set to measure the trade-off between training efficiency and model performance.

**Designed for Google Colab** - Takes advantage of Colab's GPU acceleration (recommended: T4 or A100)
**No Google Drive mounting required** - Uses Colab's session storage directly

In [1]:
import os
# Install required packages
print("Installing required packages...")

packages = [
    # "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118",
    "torch>=2.0.0",
    "torchvision>=0.15.0",
    "torchaudio>=2.0.0",
    "datasets==2.14.7",
    "huggingface-hub==0.36.2",
    "transformers==4.57.6",
    "pandas>=1.3.0",
    "numpy>=1.21.0,<1.24.0",
    "pytest>=7.0.0",
    "accelerate>=0.20.0",
    "scikit-learn>=1.0.0"
]

for package in packages:
    pkg_name = package.split()[0]
    print(f"Installing {pkg_name}...")
    # Force reinstall huggingface-hub and datasets to avoid version conflicts
    if "huggingface-hub" in package or "datasets" in package:
        os.system(f"pip install -q {package} --force-reinstall")
    else:
        os.system(f"pip install -q {package}")

print("All packages installed successfully!")

Installing required packages...
Installing torch>=2.0.0...
Installing torchvision>=0.15.0...
Installing torchaudio>=2.0.0...
Installing datasets==2.14.7...
Installing huggingface-hub==0.36.2...
Installing transformers==4.57.6...
Installing pandas>=1.3.0...
Installing numpy>=1.21.0,<1.24.0...
Installing pytest>=7.0.0...
Installing accelerate>=0.20.0...
Installing scikit-learn>=1.0.0...
All packages installed successfully!


The two cells below clone the code from GitHub and set working directories

In [2]:
# OPTION 1: Clone from GitHub and install dependencies from environment.yml

import yaml

GITHUB_REPO = "https://github.com/jtco19/ner_data_augmentation_error_analysis.git"

print("=" * 70)
print("STEP 1: Cloning repository from GitHub...")
print("=" * 70)
!git clone {GITHUB_REPO}
%cd ner_data_augmentation_error_analysis/src

print(f"Working directory: {os.getcwd()}")

STEP 1: Cloning repository from GitHub...
fatal: destination path 'ner_data_augmentation_error_analysis' already exists and is not an empty directory.
/content/ner_data_augmentation_error_analysis/src
Working directory: /content/ner_data_augmentation_error_analysis/src


In [3]:
import sys
import logging
import json
import time
from typing import Dict, Tuple
from pathlib import Path

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Find and configure path for source directories
# Works for both local execution and Colab
def find_source_dirs():
    """Find data and model directories, handling both local and Colab environments."""
    # List of possible locations to check
    search_paths = [
        Path("."),  # Current directory (local default)
        Path("./src"),  # src subfolder
        Path("/content"),  # Colab root
        Path("/content/src"),  # Colab with src folder structure
    ]
    
    data_dir = None
    model_dir = None
    base_dir = None
    
    # Search for the source directories
    for search_path in search_paths:
        if search_path.exists():
            data_candidate = search_path / "data"
            model_candidate = search_path / "model"
            
            if data_candidate.exists() and model_candidate.exists():
                data_dir = data_candidate
                model_dir = model_candidate
                base_dir = search_path
                break
    
    if data_dir is None or model_dir is None:
        raise FileNotFoundError(
            "Could not find 'data' and 'model' directories. "
            "Please ensure project files are uploaded correctly. "
            f"Checked: {[str(p) for p in search_paths]}"
        )
    
    return data_dir, model_dir, base_dir

# Find directories
data_dir, model_dir, base_dir = find_source_dirs()

# Add to path
sys.path.insert(0, str(data_dir))
sys.path.insert(0, str(model_dir))

print(f"Base directory: {base_dir}")
print(f"Data directory: {data_dir}")
print(f"Model directory: {model_dir}")
print(f"Python path configured")

Base directory: .
Data directory: data
Model directory: model
Python path configured


If the following cell throws an `ImportError`, restart the kernel and rerun the entire notebook (currently trying to identify cause of error and why it occurs randomly)

In [6]:
from load_conll2003 import load_conll2003_dataset, subsample_dataset
from bert_model import create_bert_ner_model, check_gpu_availability

print("Modules imported successfully!")

Modules imported successfully!


In [9]:
# Load CoNLL-2003 dataset directly using safe loader
print("Loading CoNLL-2003 dataset...")

# Use the inline loader that bypasses scripts
dataset = load_conll2003_dataset()

print(f"\nDataset loaded successfully!")
print(f"Dataset splits: {dataset.keys()}")
print(f"  Train: {len(dataset['train'])} samples")
print(f"  Validation: {len(dataset['validation'])} samples")
print(f"  Test: {len(dataset['test'])} samples")

# Create subsampled dataset (10% of training)
print("\nCreating 10% subsampled dataset...")

subsampled_dataset = subsample_dataset(dataset, sample_rate=0.1)

print(f"Subsampled dataset splits:")
print(f"  Train: {len(subsampled_dataset['train'])} samples")
print(f"  Validation: {len(subsampled_dataset['validation'])} samples")
print(f"  Test: {len(subsampled_dataset['test'])} samples")

Loading CoNLL-2003 dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]


Dataset loaded successfully!
Dataset splits: dict_keys(['train', 'validation', 'test'])
  Train: 14041 samples
  Validation: 3250 samples
  Test: 3453 samples

Creating 10% subsampled dataset...
Subsampled train: 14041 -> 1404 samples (10.0%)
Subsampled validation: 3250 -> 325 samples (10.0%)
Subsampled test: 3453 -> 345 samples (10.0%)
Subsampled dataset splits:
  Train: 1404 samples
  Validation: 325 samples
  Test: 345 samples


In [10]:
def train_model(
    model_name: str,
    dataset,
    num_epochs: int = 3,
    batch_size: int = 32,
    learning_rate: float = 2e-5,
    output_dir: str = None,
) -> Tuple:
    """
    Train a BERT model on the provided dataset.
    
    Args:
        model_name: Name for logging purposes
        dataset: HuggingFace DatasetDict with 'train' and 'validation' splits
        num_epochs: Number of training epochs
        batch_size: Batch size for training
        learning_rate: Learning rate for optimizer
        output_dir: Directory to save the model
        
    Returns:
        Tuple of (trained_model, training_results_dict)
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    # Create model
    model = create_bert_ner_model(num_labels=9)  # 9 NER tags
    
    # Log dataset sizes
    train_size = len(dataset["train"])
    val_size = len(dataset["validation"])
    print(f"Training samples: {train_size}")
    print(f"Validation samples: {val_size}")
    
    # Prepare dataset (tokenize and align labels)
    print("Preparing dataset...")
    train_dataset = model.prepare_dataset(dataset["train"])
    val_dataset = model.prepare_dataset(dataset["validation"])
    test_dataset = model.prepare_dataset(dataset["test"])
    
    # Train model
    print(f"Starting training for {num_epochs} epochs...")
    start_time = time.time()
    
    results = model.train(
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        output_dir=output_dir or f"./saved_models/{model_name}",
        num_epochs=num_epochs,
        per_device_batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_steps=500,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
    )
    
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")
    
    # Save model
    if output_dir:
        model.save_model(output_dir)
        print(f"Model saved to {output_dir}")
    
    return model, {
        "training_time": training_time,
        "train_samples": train_size,
        "val_samples": val_size,
    }, test_dataset

print("Training function defined.")

Training function defined.


In [11]:
def evaluate_model(model, test_dataset, model_name: str) -> Dict:
    """
    Evaluate a trained model on the test set.
    
    Args:
        model: Trained BERT model
        test_dataset: Prepared (tokenized) test dataset
        model_name: Name for logging purposes
        
    Returns:
        Dictionary with evaluation metrics
    """
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    test_size = len(test_dataset)
    print(f"Test samples: {test_size}")
    
    # Move model to CPU for evaluation (DirectML compatibility)
    model.model.to("cpu")
    
    # Evaluate
    print("Running evaluation...")
    start_time = time.time()
    
    eval_results = model.evaluate(test_dataset)
    
    eval_time = time.time() - start_time
    print(f"Evaluation completed in {eval_time:.2f} seconds")
    
    # Extract key metrics
    metrics = {
        "eval_time": eval_time,
        "test_samples": test_size,
        "eval_loss": eval_results.get("eval_loss", None),
        "eval_f1": eval_results.get("eval_f1", None),
        "eval_precision": eval_results.get("eval_precision", None),
        "eval_recall": eval_results.get("eval_recall", None),
    }
    
    return metrics

print("Evaluation function defined.")

Evaluation function defined.


In [12]:
def compare_performance(full_metrics: Dict, subsampled_metrics: Dict) -> None:
    """
    Compare performance metrics between full and subsampled models.
    
    Args:
        full_metrics: Evaluation metrics for full dataset model
        subsampled_metrics: Evaluation metrics for subsampled dataset model
    """
    print(f"\n{'='*60}")
    print("PERFORMANCE COMPARISON")
    print(f"{'='*60}\n")
    
    # Training metrics
    print("TRAINING METRICS:")
    print(f"  Full Dataset Model:")
    print(f"    - Training samples: {full_metrics.get('train_samples', 'N/A')}")
    print(f"    - Training time: {full_metrics.get('training_time', 'N/A'):.2f}s")
    
    print(f"  Subsampled (10%) Model:")
    print(f"    - Training samples: {subsampled_metrics.get('train_samples', 'N/A')}")
    print(f"    - Training time: {subsampled_metrics.get('training_time', 'N/A'):.2f}s")
    
    time_reduction = (
        (full_metrics.get('training_time', 0) - subsampled_metrics.get('training_time', 0)) /
        full_metrics.get('training_time', 1) * 100
    )
    print(f"  Time reduction: -{time_reduction:.1f}%\n")
    
    # Evaluation metrics
    print("EVALUATION METRICS:")
    print(f"  Full Dataset Model:")
    print(f"    - Eval Loss: {full_metrics.get('eval_loss', 'N/A'):.4f}")
    print(f"    - F1 Score: {full_metrics.get('eval_f1', 'N/A'):.4f}")
    print(f"    - Precision: {full_metrics.get('eval_precision', 'N/A'):.4f}")
    print(f"    - Recall: {full_metrics.get('eval_recall', 'N/A'):.4f}")
    
    print(f"  Subsampled (10%) Model:")
    print(f"    - Eval Loss: {subsampled_metrics.get('eval_loss', 'N/A'):.4f}")
    print(f"    - F1 Score: {subsampled_metrics.get('eval_f1', 'N/A'):.4f}")
    print(f"    - Precision: {subsampled_metrics.get('eval_precision', 'N/A'):.4f}")
    print(f"    - Recall: {subsampled_metrics.get('eval_recall', 'N/A'):.4f}")
    
    # Compute differences
    print("\nPERFORMANCE DELTA (Subsampled vs Full):")
    
    if full_metrics.get('eval_f1') and subsampled_metrics.get('eval_f1'):
        f1_delta = subsampled_metrics['eval_f1'] - full_metrics['eval_f1']
        print(f"  - F1 Score Delta: {f1_delta:+.4f} ({f1_delta/full_metrics['eval_f1']*100:+.2f}%)")
    
    if full_metrics.get('eval_precision') and subsampled_metrics.get('eval_precision'):
        prec_delta = subsampled_metrics['eval_precision'] - full_metrics['eval_precision']
        print(f"  - Precision Delta: {prec_delta:+.4f} ({prec_delta/full_metrics['eval_precision']*100:+.2f}%)")
    
    if full_metrics.get('eval_recall') and subsampled_metrics.get('eval_recall'):
        rec_delta = subsampled_metrics['eval_recall'] - full_metrics['eval_recall']
        print(f"  - Recall Delta: {rec_delta:+.4f} ({rec_delta/full_metrics['eval_recall']*100:+.2f}%)")
    
    print(f"\n{'='*60}\n")

print("Comparison function defined.")

Comparison function defined.


In [13]:
# Create output directories
full_model_dir = "./saved_models/bert_full_conll2003"
subsampled_model_dir = "./saved_models/bert_subsampled_10pct_conll2003"

os.makedirs(full_model_dir, exist_ok=True)
os.makedirs(subsampled_model_dir, exist_ok=True)

# Train full dataset model
print(f"\n{'='*60}")
print("STARTING FULL DATASET TRAINING")
print(f"{'='*60}")

full_model, full_train_metrics, test_dataset_full = train_model(
    model_name="bert-base-cased",
    dataset=dataset,
    num_epochs=3,
    batch_size=32,
    learning_rate=2e-5,
    output_dir=full_model_dir,
)

print("Full dataset training complete!")


STARTING FULL DATASET TRAINING

Training: bert-base-cased


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Tokenizing and aligning labels:   0%|          | 0/3250 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/3453 [00:00<?, ? examples/s]

Starting training for 3 epochs...


Epoch,Training Loss,Validation Loss
1,0.145400,0.104316
2,0.071200,0.069343
3,0.035000,0.059342


Training completed in 1133.59 seconds
Model saved to ./saved_models/bert_full_conll2003
Full dataset training complete!


In [14]:
# Train subsampled model
print(f"\n{'='*60}")
print("STARTING SUBSAMPLED (10%) DATASET TRAINING")
print(f"{'='*60}")

subsampled_model, subsampled_train_metrics, test_dataset_subsampled = train_model(
    model_name="BERT on 10% Subsampled CoNLL-2003 Training Set",
    dataset=subsampled_dataset,
    num_epochs=3,
    batch_size=32,
    learning_rate=2e-5,
    output_dir=subsampled_model_dir,
)

print("Subsampled dataset training complete!")


STARTING SUBSAMPLED (10%) DATASET TRAINING

Training: BERT on 10% Subsampled CoNLL-2003 Training Set


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training samples: 1404
Validation samples: 325
Preparing dataset...


Tokenizing and aligning labels:   0%|          | 0/1404 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/325 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/345 [00:00<?, ? examples/s]

Starting training for 3 epochs...


Epoch,Training Loss,Validation Loss
1,No log,1.922934
2,No log,0.983681
3,1.728500,0.552244


Training completed in 150.60 seconds
Model saved to ./saved_models/bert_subsampled_10pct_conll2003
Subsampled dataset training complete!


When prompted in the cell below, enter "3" to avoid visualizing metrics (avoids needing a W&B account)

In [15]:
# Evaluate both models
print(f"\n{'='*60}")
print("EVALUATING MODELS")
print(f"{'='*60}")

full_eval_metrics = evaluate_model(
    full_model, test_dataset_full, model_name="Full Model Evaluation"
)

subsampled_eval_metrics = evaluate_model(
    subsampled_model,
    test_dataset_subsampled,  # Use same test set for fair comparison
    model_name="Subsampled Model Evaluation",
)

# Combine metrics
full_metrics = {**full_train_metrics, **full_eval_metrics}
subsampled_metrics = {**subsampled_train_metrics, **subsampled_eval_metrics}

print("Evaluation complete!")


EVALUATING MODELS

Evaluating: Full Model Evaluation
Test samples: 3453
Running evaluation...


/content/ner_data_augmentation_error_analysis/src/model/bert_model.py:463: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Evaluation completed in 54.15 seconds

Evaluating: Subsampled Model Evaluation
Test samples: 345
Running evaluation...


/content/ner_data_augmentation_error_analysis/src/model/bert_model.py:463: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation completed in 3.43 seconds
Evaluation complete!


In [16]:
# Compare performance
compare_performance(full_metrics, subsampled_metrics)

# Save results to JSON
results = {
    "full_model": full_metrics,
    "subsampled_model": subsampled_metrics,
}

results_dir = "./results"
os.makedirs(results_dir, exist_ok=True)

results_path = os.path.join(results_dir, "baseline_experiment.json")

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path}")
print("\n✅ Experiment completed successfully!")

# Display results
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(json.dumps(results, indent=2))


PERFORMANCE COMPARISON

TRAINING METRICS:
  Full Dataset Model:
    - Training samples: 14041
    - Training time: 1133.59s
  Subsampled (10%) Model:
    - Training samples: 1404
    - Training time: 150.60s
  Time reduction: -86.7%

EVALUATION METRICS:
  Full Dataset Model:
    - Eval Loss: 0.1713
    - F1 Score: 0.9699
    - Precision: 0.9710
    - Recall: 0.9692
  Subsampled (10%) Model:
    - Eval Loss: 0.6400
    - F1 Score: 0.7098
    - Precision: 0.7163
    - Recall: 0.7696

PERFORMANCE DELTA (Subsampled vs Full):
  - F1 Score Delta: -0.2601 (-26.81%)
  - Precision Delta: -0.2547 (-26.23%)
  - Recall Delta: -0.1996 (-20.59%)


Results saved to ./results/baseline_experiment.json

✅ Experiment completed successfully!

FINAL RESULTS SUMMARY
{
  "full_model": {
    "training_time": 1133.5933558940887,
    "train_samples": 14041,
    "val_samples": 3250,
    "eval_time": 54.14665961265564,
    "test_samples": 3453,
    "eval_loss": 0.17133884131908417,
    "eval_f1": 0.9699064012070